In [1]:
import os
import cv2
import numpy as np
import random
import shutil

# 原始分類資料夾
SRC_DIR = r"C:\temp\classification_FINAL_ROI\train"

# 輸出擴增後資料夾
DST_DIR = r"C:\temp\classification_FINAL_ROI_AUG\train"
os.makedirs(DST_DIR, exist_ok=True)

# 要做的擴增數量（每張圖擴增幾張）
AUG_PER_IMAGE = 3

# 增強工具函式
def rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    return cv2.warpAffine(img, M, (w, h), borderValue=(0, 0, 0))

def add_noise(img):
    noise = np.random.normal(0, 2, img.shape).astype(np.int16)
    img = img.astype(np.int16) + noise
    return np.clip(img, 0, 255).astype(np.uint8)


def blur(img):
    return cv2.GaussianBlur(img, (5, 5), 0)

# 擴增處理每個類別資料夾
for cls in ['benign', 'malignant']:
    src_cls = os.path.join(SRC_DIR, cls)
    dst_cls = os.path.join(DST_DIR, cls)
    os.makedirs(dst_cls, exist_ok=True)

    imgs = [f for f in os.listdir(src_cls) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for fname in imgs:
        src_path = os.path.join(src_cls, fname)
        img = cv2.imread(src_path)

        # 複製原始圖
        shutil.copy(src_path, os.path.join(dst_cls, fname))

        for i in range(AUG_PER_IMAGE):
            aug = img.copy()

            if random.random() < 0.5:
                aug = cv2.flip(aug, 1)  # 水平翻轉

            if random.random() < 0.9:
                angle = random.uniform(-20, 20)
                aug = rotate(aug, angle)

            if random.random() < 0.5:
                aug = add_noise(aug)

            if random.random() < 0.5:
                aug = blur(aug)

            new_name = fname.replace('.', f'_aug{i}.')
            cv2.imwrite(os.path.join(dst_cls, new_name), aug)

    print(f"✅ {cls} 完成擴增，共處理 {len(imgs)} 張圖片")

print("🎉 所有分類影像擴增完成！")


✅ benign 完成擴增，共處理 162 張圖片
✅ malignant 完成擴增，共處理 171 張圖片
🎉 所有分類影像擴增完成！
